In [1]:
pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [4]:
faturas_csv = ['../DadosFaturas/Fatura_2025-03-20.csv', '../DadosFaturas/Fatura_2025-04-20.csv', '../DadosFaturas/Fatura_2025-05-20.csv', 
               '../DadosFaturas/Fatura_2025-06-20.csv', '../DadosFaturas/Fatura_2025-07-20.csv', '../DadosFaturas/Fatura_2025-08-20.csv',
               '../DadosFaturas/Fatura_2025-09-20.csv', '../DadosFaturas/Fatura_2025-10-20.csv', '../DadosFaturas/Fatura_2025-11-20.csv',
               '../DadosFaturas/Fatura_2025-12-20.csv', '../DadosFaturas/Fatura_2026-01-20.csv', '../DadosFaturas/Fatura_2026-02-20.csv']

dados_faturas_list = []

for f in faturas_csv:
    fatura = pd.read_csv(f, sep=';')
    dados_faturas_list.append(fatura)

dados_faturas = pd.concat(dados_faturas_list)

In [5]:
dados_faturas.rename(columns={
    'Nome no Cartão': 'nome_cartao', 
    'Final do Cartão': 'final_cartao',
    'Data de Compra': 'data_completa',
    'Categoria': 'nome_categoria', 
    'Descrição': 'nome_estabelecimento',
    'Valor (em R$)': 'valor_brl', 
    'Valor (em US$)': 'valor_usd', 
    'Parcela': 'parcela_texto', 
    'Cotação (em R$)': 'cotacao_brl'
}, inplace=True)

In [6]:
print("Linhas carregadas:", len(dados_faturas))

Linhas carregadas: 1945


In [7]:
# Função de limpeza e padronização
def limpeza(colunas):
    for col in colunas:
        dados_faturas[col] = (
            dados_faturas[col]
            .astype(str) # Converte para String
            .str.strip() # Remove espaços em branco no começo e no final da String
            .str.upper() # Converte toda a String para maiúsculas
            .str.replace(r'\s+', ' ', regex=True) # Substitui qualquer sequência de espaços por apenas um espaço simples.
        )

In [8]:
# DIM titular

colunas_titular = ["nome_cartao", "final_cartao"]

limpeza(colunas_titular)

dim_titular = dados_faturas[colunas_titular].drop_duplicates().reset_index(drop=True)
dim_titular["id_titular"] = dim_titular.index + 1
dim_titular = dim_titular[["id_titular", "nome_cartao", "final_cartao"]]

# Se o número de IDs únicos for igual ao tamanho do DataFrame, deu certo.
print(dim_titular['id_titular'].nunique() == len(dim_titular))
print("titulares:", len(dim_titular))
print(dim_titular.head())

True
titulares: 5
   id_titular      nome_cartao final_cartao
0           1       VIN DIESEL         1115
1           2       VIN DIESEL         1122
2           3  CHARLIZE THERON         1153
3           4      BRIAN TYLER         1114
4           5       EVA MENDES         1117


In [9]:
# DIM categoria

colunas_categoria = ["nome_categoria"]

limpeza(colunas_categoria)

dados_faturas["nome_categoria"] = dados_faturas["nome_categoria"].str.replace("-", "Não categorizado")

dim_categoria = dados_faturas[colunas_categoria].drop_duplicates().reset_index(drop=True)
dim_categoria["id_categoria"] = dim_categoria.index + 1
dim_categoria = dim_categoria[["id_categoria", "nome_categoria"]]

# Se o número de IDs únicos for igual ao tamanho do DataFrame, deu certo.
print(dim_categoria['id_categoria'].nunique() == len(dim_categoria))
print("Categorias:", len(dim_categoria))
print(dim_categoria.head())

True
Categorias: 36
   id_categoria                                     nome_categoria
0             1                            DEPARTAMENTO / DESCONTO
1             2                  ASSISTÊNCIA MÉDICA E ODONTOLÓGICA
2             3  SUPERMERCADOS / MERCEARIA / PADARIAS / LOJAS D...
3             4                          RELACIONADOS A AUTOMOTIVO
4             5                     RESTAURANTE / LANCHONETE / BAR


In [10]:
# DIM estabelecimento

colunas_estabelecimento = ["nome_estabelecimento"]

limpeza(colunas_estabelecimento)

dim_estabelecimento = dados_faturas[colunas_estabelecimento].drop_duplicates().reset_index(drop=True)
dim_estabelecimento["id_estabelecimento"] = dim_estabelecimento.index + 1
dim_estabelecimento = dim_estabelecimento[["id_estabelecimento", "nome_estabelecimento"]]

# Se o número de IDs únicos for igual ao tamanho do DataFrame, deu certo.
print(dim_estabelecimento['id_estabelecimento'].nunique() == len(dim_estabelecimento))
print("Estabelecimentos:", len(dim_estabelecimento))
print(dim_estabelecimento.head())

True
Estabelecimentos: 426
   id_estabelecimento  nome_estabelecimento
0                   1          HUB*NETSHOES
1                   2           OPTICA RUDI
2                   3              COTRIJAL
3                   4     RECUPERADORADEPAR
4                   5  EPIC MOTORS MECANICA


In [11]:
# DIM data

# Converter coluna para data
dados_faturas['data_completa'] = pd.to_datetime(dados_faturas['data_completa'], format="%d/%m/%Y")

# Data mínima e máxima das transacões
data_inicio = dados_faturas['data_completa'].min()
data_fim = dados_faturas['data_completa'].max()

# Sequência contínua de dias (sem buracos)
dias_continuos = pd.date_range(start=data_inicio, end=data_fim, freq='D')

# Transforme essa sequência no seu DataFrame de Dimensão
dim_data = pd.DataFrame({'data': dias_continuos})

# Criando o id_data no formato AAAAMMDD
dim_data['id_data'] = dim_data['data'].dt.strftime('%Y%m%d').astype(int)

# Os atributos da dimensão
dim_data['ano'] = dim_data['data'].dt.year
dim_data['mes'] = dim_data['data'].dt.month
dim_data['dia'] = dim_data['data'].dt.day
dim_data['dia_semana'] = dim_data['data'].dt.day_name()
dim_data['trimestre'] = dim_data['data'].dt.quarter

# Reorganizando as colunas
dim_data = dim_data[['id_data', 'data', 'ano', 'mes', 'dia', 'dia_semana', 'trimestre']]

print(dim_data.head())

    id_data       data   ano  mes  dia dia_semana  trimestre
0  20241012 2024-10-12  2024   10   12   Saturday          4
1  20241013 2024-10-13  2024   10   13     Sunday          4
2  20241014 2024-10-14  2024   10   14     Monday          4
3  20241015 2024-10-15  2024   10   15    Tuesday          4
4  20241016 2024-10-16  2024   10   16  Wednesday          4


In [12]:
# Merge com dim titular
df_temp = pd.merge(
    left=dados_faturas,
    right=dim_titular[["id_titular", "nome_cartao", "final_cartao"]],
    on=["nome_cartao", "final_cartao"],
    how="left",
    validate="m:1"
)

In [13]:
# Merge com dim categoria
df_temp = pd.merge(
    left=df_temp,
    right=dim_categoria[["id_categoria", "nome_categoria"]],
    on="nome_categoria",
    how="left",
    validate="m:1"
)

In [14]:
# Merge com dim estabelecimento
df_temp = pd.merge(
    left=df_temp,
    right=dim_estabelecimento[["id_estabelecimento", "nome_estabelecimento"]],
    on="nome_estabelecimento",
    how="left",
    validate="m:1"
)

In [15]:
# Transformar a data_completa no formato AAAAMMDD númerico
df_temp['id_data'] = pd.to_datetime(df_temp['data_completa']).dt.strftime('%Y%m%d').astype(int)

In [16]:
# Criaçaõ da fato

colunas_fato = [
    "valor_brl", "valor_usd", "cotacao_brl",
    "parcela_texto", "id_titular", "id_categoria",
    "id_estabelecimento", "id_data"
]

fato_transacao = df_temp[colunas_fato].copy()

fato_transacao.loc[fato_transacao["parcela_texto"] == "Única", "parcela_texto"] = "1/1"
fato_transacao["num_parcela"] = fato_transacao["parcela_texto"].str.split("/").str[0].astype(int)
fato_transacao["total_parcelas"] = fato_transacao["parcela_texto"].str.split("/").str[1].astype(int)

fato_transacao = fato_transacao.reset_index(drop=True)
fato_transacao["id_transacao"] = fato_transacao.index + 1

print("Transacoes: ", len(fato_transacao))
print(fato_transacao.head())
print(fato_transacao.dtypes)

del df_temp

Transacoes:  1945
   valor_brl  valor_usd  cotacao_brl parcela_texto  id_titular  id_categoria  \
0      52.99        0.0          0.0          5/10           1             1   
1     162.00        0.0          0.0          5/10           1             2   
2      48.49        0.0          0.0          4/10           1             3   
3     498.08        0.0          0.0           4/4           1             4   
4     633.33        0.0          0.0           2/3           1             4   

   id_estabelecimento   id_data  num_parcela  total_parcelas  id_transacao  
0                   1  20241012            5              10             1  
1                   2  20241028            5              10             2  
2                   3  20241116            4              10             3  
3                   4  20241126            4               4             4  
4                   5  20250124            2               3             5  
valor_brl             float64
valor_usd

In [17]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("../.env")

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port= os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [18]:
#Verifica se todos os IDs de titular nas transações existem na tabela titulares.
print(fato_transacao["id_titular"].isin(dim_titular["id_titular"]).all())

#Confere se todas as datas referenciadas existem na tabela datas.
print(fato_transacao["id_data"].isin(dim_data["id_data"]).all())

#Confere se todas as categorias referenciadas existem na tabela categorias
print(fato_transacao["id_categoria"].isin(dim_categoria["id_categoria"]).all())

#Confere se todos os estabelecimentos referenciados existem na tabela estabelecimentos
print(fato_transacao["id_estabelecimento"].isin(dim_estabelecimento["id_estabelecimento"]).all())

#Se aparecer número > 0, significa que existem transações duplicadas.
print(fato_transacao.duplicated().sum())

#Saída esperada
#True
#True
#True
#True
#0

True
True
True
True
0


In [19]:
dim_titular.to_sql("dim_titular", engine, if_exists="append", index=False)

5

In [20]:
dim_categoria.to_sql("dim_categoria", engine, if_exists="append", index=False)

36

In [21]:
dim_data.to_sql("dim_data", engine, if_exists="append", index=False)

488

In [22]:
dim_estabelecimento.to_sql("dim_estabelecimento", engine, if_exists="append", index=False)

426

In [23]:
fato_transacao.to_sql("fato_transacao", engine, if_exists="append", index=False)

945

In [24]:
from sqlalchemy import text

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM dim_titular")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM dim_categoria")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM dim_data")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM dim_estabelecimento")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM fato_transacao")).fetchone())

(5,)
(36,)
(488,)
(426,)
(1945,)
